#Spark SQL
from pysprk.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.read. 

##before we create a df or RDD we need to create the sparkSession object by instatntiating sparkSession class (by default databricks create the sparksession if we create a notebook)
###SparkSession is the entry point for accessing the cluster through spark
getorcreate will import the libreries and cretae the session or create the new one

In [0]:
from pyspark.sql.session import SparkSession
print(spark) #already instantiated by databricks
spark1=SparkSession.builder.getOrCreate()
print(spark1)#we instantiated


##create a dbfs volume and upload the data in that volume
##what are the other file sysytem URI (unique resource indentity)available
file:///, hdfs:///, dbfs:///, gs:///, s3:///



###how to read the data from the file system and load it into the distrubution memory for further processing/load - using different methods/options from different sources (fs & db) and different builtin formats like csv,json,orc,parquet,delta,tables



In [0]:
#if i dont use the any function in csv function what will it use by default functionality?
#1. by default it will take ',' as a delimiter
#2. by default it will take _c0,_c1_...cn it will apply as column header
#3. it will treat all the column as string

csv_df1=spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/custs")
display(csv_df1) #it will show the output in table format
#csv_df1.show(2) # will produce output in dataframe format

##Sampledata
4000001	Kristina	Chung	55	Pilot
4000002	Paige	Chen	77	Teacher
4000003	Sherri	Melton	34	Firefighter



In [0]:
#1. header concept
#
csv_df1=spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/custs").toDF("id","fname","lname","age","prof")
csv_df1.show(1)
csv_df2=spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/custs_header", header=True)
csv_df2.show(2)


In [0]:
#printing schema (equivalent to describe schema)
csv_df1.printSchema()
csv_df2.printSchema()

In [0]:
#3. inferring Schema (use this with care not good for using t under pre-defined schema on dataset)
csv_df1=spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/custs", inferSchema=True).toDF("id","fname","lname","age","prof").printSchema()

In [0]:
csv_df1=spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/custs_header_oth_del", header=True, sep='~').show(3)

In [0]:
#we can also use the option separately as below 
#csv_df2= spark.read.option("header","True").option("inferSchema","True").option("sep","~").csv("/Volumes/workspace/wd36schema/ingestion_volume/source/custs_header_oth_del").show(3)
#also we can go as below instead of using option multiple time
#csv_df2= spark.read.options(header="True", inferSchema="True", sep="~").csv("/Volumes/workspace/wd36schema/ingestion_volume/source/custs_header_oth_del").show(3)


In [0]:
#read data from multiple files
csv_df1 = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/cust*", header=True, sep='~')
csv_df1.show()
print(csv_df1.count())

In [0]:
dfjson= spark.read.json("/Volumes/workspace/wd36schema/ingestion_volume/source/simple_json.txt")
dfjson.printSchema()
dfjson.show()
##here by default it will take inferschema as true

In [0]:
str1= "id int, name string, amt double, dop string"
dfjson= spark.read.schema(str1).json("/Volumes/workspace/wd36schema/ingestion_volume/source/simple_json.txt")
dfjson.printSchema()
dfjson.show()

In [0]:
#primitiveAsString - it inferSchema as false, where it will be true by default 
dfjson2= spark.read.json("/Volumes/workspace/wd36schema/ingestion_volume/source/simple_json.txt", primitiveAsString=True)
dfjson2.printSchema()
dfjson2.show()

In [0]:
Source1df = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/source1.txt", header=True, inferSchema=True)
Source1df.write.orc("/Volumes/workspace/wd36schema/ingestion_volume/source/orcoutput", mode="append")
Source2df= spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/source2.txt", header=True, inferSchema=True)
Source2df.write.orc("/Volumes/workspace/wd36schema/ingestion_volume/source/orcoutput", mode="append")
Source3df= spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/source3.txt", header=True, inferSchema=True)
Source3df.write.orc("/Volumes/workspace/wd36schema/ingestion_volume/source/orcoutput", mode="append")

                    

In [0]:
#read the data from source and do the schema evolution and get the datafrome evolved
post_day= spark.read.orc("/Volumes/workspace/wd36schema/ingestion_volume/source/orcoutput/", mergeSchema= True)
display(post_day)

#CSV advanced write options

In [0]:
df1=spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/custs", header=False, inferSchema=True)
df1.write.csv(path="/Volumes/workspace/wd36schema/ingestion_volume/source/csvoutput", mode= 'overwrite', compression= 'None', sep=',', header=False, escape= '~', quote= "'")

display(df1)

#joins
![image_1772968380974.png](./image_1772968380974.png "image_1772968380974.png")

![image_1773031298128.png](./image_1773031298128.png "image_1773031298128.png")

![image_1773034767059.png](./image_1773034767059.png "image_1773034767059.png")


In [0]:
innerjoindf = leftdf.join(rightdf, on='id', how='inner')
#this is the most commonly used syntax for inner join in pyspark

In [0]:
#this way is also fine but in the real world most commonly ised python syntax is the above one

df = leftdf.join(rightdf, ["dept_id"], "inner")

In [0]:
df.join(df2, on= "cust_id"=="cid", how= "inner")

#joins representation in ven diagram
![image_1773037691186.png](./image_1773037691186.png "image_1773037691186.png")


#special/optimized joins
![image_1773037911723.png](./image_1773037911723.png "image_1773037911723.png")

![image_1773117376575.png](./image_1773117376575.png "image_1773117376575.png")

![image_1773128205616.png](./image_1773128205616.png "image_1773128205616.png")

![image_1773129776770.png](./image_1773129776770.png "image_1773129776770.png")

![image_1773138128354.png](./image_1773138128354.png "image_1773138128354.png")

![image_1773147188193.png](./image_1773147188193.png "image_1773147188193.png")

In [0]:
#window function for top n analysis

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

windowSpec = Window.partitionBy("cust_id").orderBy(("txndt"))
newdf = orderjoindf.withColumn("seqnum", row_number().over(windowSpec))
#most preferred way is to define windowSpec separately and reuse it

#Alternatively, you can inline the window specification, but it's less readable and reusable
newdf = orderjoindf.withColumn("seqnum", row_number().over(Window.partitionBy("cust_id").orderBy(("txndt")))).filter("seqnum <= 3")

In [0]:
#Comparitive analytics - lead & lag
sk_orderjoinedf=orderjoineddf.withColumn ("nexttransamt", lead("amt",1, -1).over(Window.partitionBy("custid").orderBy(asc("txndt")))).\
withColumn("priortransamt", lag("amt",1, -1).over(Window.partitionBy("custid").orderBy(asc("txndt"))))
#display(sk_orderjoinedf)
pattern_df=sk_orderjoinedf.\
withColumn ("transpattern"
when ((col("priortransamt")==-1),"first transaction").\ 
when (col("amt") >col("priortransamt"),"increase").\
when (col ("amt")<col("priortransamt"),"decrease"). \
otherwise("same"))
display(pattern_df)

#we can also write it as
windowspec= Window.partitionBy("custid").orderBy(asc("txndt"))
sk_orderjoinedf= orderjoineddf.withColumn ("nexttransamt", lead("amt",1, -1).over(windowspec).\
 withColumn("priortransamt", lag("amt",1, -1).over(windowspec))
#display(sk_orderjoinedf)
pattern_df=sk_orderjoinedf.\
withColumn ("transpattern"
when ((col("priortransamt")==-1),"first transaction").\ 
when (col("amt") >col("priortransamt"),"increase").\
when (col ("amt")<col("priortransamt"),"decrease"). \
otherwise("same"))
display(pattern_df)

[Uploading image_1773297445623.png...]

if we need to increase the table horizontally we can use join strategies but if we want to increase the table vertically then we can use set operations


![image_1773496950053.png](./image_1773496950053.png "image_1773496950053.png")
